# **Project title:** Quantitative assessment of slc35a2 localization to the Golgi in single cells
### **Researcher:** Xavier Williams
### **Notebook author:** Shannon Rhoads (srhoads@unc.edu)

### **Editing & version log:**
- v0.1: 20260115 - downloaded [notebook 1.1_infer_masks_from-composite_with_nuc.ipynb](https://github.com/SCohenLab/infer-subc/blob/v2.0.0b1/notebooks/part_1_segmentation_workflows/1.1_infer_masks_from-composite_with_nuc.ipynb) from [infer-subc version v2.0.0b1](https://github.com/SCohenLab/infer-subc/tree/v2.0.0b1)
- v0.2: 20260115 - the following edits were made:
    - Update image reader to BioImageIO package
    - Nuclei declumping methods were added to better separate neighboring nuclei
    - Filtering step to remove nuclei that are extremely bright (these are usually rounded and likely dividing or dying)

----------
## **How to use Jupyter notebooks**

Advance through each block of code below by pressing `Shift`+`Enter` or pressing the "Execute Cell" (`▶️`) button to the left of each block.

You will see a series of instructions before each block of code. Be on the look out for the following headers and follow the instructions accordingly:
- &#x1F3C3; **Run code; no user input required** - proceed without adding anything to the code block
- &#x1F453; **FYI** (for your information) - helpful information usually to bring context to what is going on
- &#x1F6D1; &#x270D; **User Input Required** - stop and input the appropriate information about your data. The following indicator will also be present in the code block:
   ```python 
   #### USER INPUT REQUIRED ###
   ```

----------

----------
# **Start of analysis pipeline:**


### **Notebook goals:**
To determine the most appropriate settings to segment the nucleus and cell mask from fluorescence microscopy images. These settings will be applied to a batch of images in the batch_process_segmentation notebook.

### 👣 **Summary of analysis steps**  

➡️ **INPUT**

In this workflow, a single or multi-channel confocal microscopy image of fluorescently tagged organelles will be used to "infer" (or segment) the cell and nucleus masks.

**EXTRACTION**
- **`STEP 1`** - Segment nuclei

    - (A) select single channel containing the nuclei marker (channel number = user input)
    - (B) rescale intensity of composite image (min=0, max=1); apply median (median size = user input) and gaussian filter (sigma = user input)
    - (C) log transform image and calculate Li's minimum cross entropy threshold value; apply threshold to image (thresholding options = user input)
    - (D) fill holes (hole size = user input) and remove small objects (object size = user input)
    - (E) label nuclei objects
    - (F) declump nuclei objects

- **`STEP 2`** - Create composite image

    - determine weight to apply to each channel of the intensity image (w# = user input)
    - rescale summed image intensities (rescale = user input)

**PRE-PROCESSING**
- **`STEP 3`** - Rescale and smooth image

    - rescale intensity of composite image (min=0, max=1)
    - median filter (median size = user input)
    - gaussian filter (sigma = user input)

- **`STEP 4`** Log transform + Scharr edge detection

    - log transform image
    - apply scharr edge detection filter 
    - combine log image + scharr edge filtered intensity

**CORE PROCESSING**
- **`STEP 5`** Global + local thresholding (AICSSeg – MO)

    - apply MO thresholding method from the Allen Cell [aicssegmentation](https://github.com/AllenCell/aics-segmentation) package (threshold options = user input)

**POST-PROCESSING**
- **`STEP 6`** Remove small holes and objects

    - fill holes (hole size = user input)
    - remove small objects (object size = user input)
    - filter method (method = user input)

**POST-POST-PROCESSING**
- **`STEP 7`** Select one cellmask/nuclei based on signal

    - label unique cell objects based on watershed seeded from the nuclei objects

**EXPORT** ➡️
- **`STEP 8`** - Stack masks

    - stack masks in order of **nucleus**, **cellmask** and **cytoplasm** mask

---------------------
## **IMPORTS AND LOAD IMAGE**
&#x1F453; **FYI:** *This section loads a single image from a folder of data that you specify. This image will be used to optimize the segmentation settings you'd like to apply across the entire dataset of images. Follow the prompts below for specifics on each step. More details about some of the functions included in this subsection are outlined in the`infer-subc` [1.0_image_setup](https://github.com/SCohenLab/infer-subc/blob/v2.0.0b1/notebooks/part_1_segmentation_workflows/1.0_image_setup.ipynb) notebook. Please visit that notebook first if you are confused about any of the code included here.*

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following information about your data: 
- `path_to_repository`: the path to the local golgi-coloc-analysis repository folder as a string

In [ ]:
#### USER INPUT REQUIRED ###
path_to_repository = r"C:\Users\Shannon\Documents\Python_Scripts\Golgi-coloc-analysis"

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** The block of code below imports all of the packages, or specific functions, that you are planning to utilize in this notebook. If it is not imported here, you will not be able to use it below.

In [ ]:
from pathlib import Path
import os
from typing import Union

import numpy as np
import pandas as pd
import napari
from napari.utils.notebook_display import nbscreenshot

from skimage.morphology import binary_erosion, binary_opening
from skimage.segmentation import watershed
from skimage.feature import peak_local_max, canny
import scipy.ndimage as ndi

from infer_subc.core.file_io import (read_czi_image,
                                     export_inferred_organelle,
                                     list_image_files)
from infer_subc.core.img import *
from infer_subc.organelles import (choose_max_label_cellmask_union_nucleus,
                                   non_linear_cellmask_transform)

from bioio import BioImage
from skimage.measure import regionprops, regionprops_table
import tifffile

import sys
sys.path.insert(0, path_to_repository)
from src.segmentation import filter_nuc_by_intensity, infer_nuclei_fromlabel, infer_cellmask_fromcomposite, infer_masks


%load_ext autoreload
%autoreload 2

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following information about your data: 
- `im_type`: the file type of your image; this must be written within quotation markers. 
    - EX: ".czi" or ".tiff"
- `data_root_path`: the path to the folder that your input data and a separate folder for segmentation outputs; this must be written within quotation markers. If you are using a Windows computer, paths are copied with backslashes ("\"). Python will be confused by this; switch these to forward slashes ("/") or place an "r" in front of the path string.  
    - EX: "C:/Users/{your-user-name}/Documents/Exp1" or r"C:\Users\\{your-user-name}\Documents\Exp1"
- `raw_data_path`: the path to the folder that contains your input data; this must be written within quotation markers. 
    - EX: data_root_path / "input"
- `seg_data_path`: the path to the folder where segmentation output files will be saved; this must be written within quotation markers. EX: data_root_path / "segmentations". If this folder does not exist already, it will be created for you in the subsequent lines of code.

In [ ]:
#### USER INPUT REQUIRED ###
im_type = ".czi"
data_root_path = Path(os.path.dirname(os.getcwd())) 
raw_data_path = Path(data_root_path) / "pilot-data"
seg_data_path = Path(data_root_path) / "test-segmentation"

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** A list of the images included in the `raw_data_path` folder is printed below for easy reference below.

In [ ]:
# Create the output directory to save the segmentation outputs in.
if not Path.exists(seg_data_path):
    Path.mkdir(seg_data_path)
    print(f"Segmentation data path did not exist; making it now: {seg_data_path}")

# list files in the input folder
print(f"Listing all {im_type} files in the raw data path: ")
img_file_list = list_image_files(Path(raw_data_path),im_type)
pd.set_option('display.max_colwidth', None)
pd.DataFrame({"Image Name":img_file_list})

#### &#x1F6D1; &#x270D; **User Input Required:**

Use the list above to specify which image you wish to analyze based on its index: 
- `test_img_n`: the index, or number, associated with your image of choice. This value can be found to the left of the image path listed in the table above.
- `channel_names`: specify the name of each channel in your image in order. The name of each channel should be within quotation markers and separated by a common. The entire list should be within square brackets ([ ]) -- a Python list.
    - EX: ['DAPI', 'slc35a2', 'golgi']
    - In the step below, this list will be used to interpret the channel order, so they should be listed in order.

In [ ]:
#### USER INPUT REQUIRED ###
test_img_n = 0
channel_names = ['golgi', 'slc35a2', 'DAPI']

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block reads the image and image metadata into memory. The metadata is printed and the image is visualized in Napari. A new window will appear displaying the Napari Viewer.

*IMPORTANT NOTES:*
1. *The code block below is "hard-coded" to process 3D images.* 
2. *Metadata is extracted correctly based on the .czi files. If file type changes, the metadata structure may also change in impact the values below. The most important variables to update from those below are the `dim_size` which is used to derive the scale (voxel size) and the `channel_axis`.*
3. *If you image type is a .czi image series (multiple FOVs in one file), the image import method used below will ONLY READS THE FIRST FOV.*

In [ ]:
# read in image and metadata
print(f"Reading in image and metadata for {img_file_list[test_img_n]}...")

file_path = str(img_file_list[test_img_n])
raw_file = BioImage(file_path)
img_data = np.squeeze(raw_file.data) # np.squeeze removes single-dimensional entries from the shape of an array
ome_metadata = raw_file.ome_metadata

# extra metadata information
dimensions = raw_file.standard_metadata.dimensions_present
channel_axis = str(dimensions).index('C')
dim_size = {"Z": raw_file.physical_pixel_sizes.Z, 
            "Y": raw_file.physical_pixel_sizes.Y, 
            "X": raw_file.physical_pixel_sizes.X}
# select the scale values from the dim_size dict in the order in which X, Y, and Z appear in the dimensions string
scale_order = [dim for dim in str(dimensions) if dim in ['Z','Y','X']]
scale = tuple([dim_size[dim] for dim in scale_order])

meta_dict = {'processing': "Masks (cell/nucleus) segmentation from https://github.com/shanrhoads/PNN-morpho-quant", 
             'file_name': file_path, 
             'scale': scale}

print("Metadata information")
print(f"File path: {file_path}")
for i in list(range(len(channel_names))):
    print(f"Channel {i} name: {channel_names[i]}")
print(f"Scale ({scale_order}): {scale}")
print(f"Channel axis: {channel_axis}")

# open viewer and add images
viewer = napari.Viewer()
for i, channel_name in enumerate(channel_names):
    viewer.add_image(img_data[i],
                     scale=scale,
                     name=f"Channel {i}: {channel_name}")
viewer.grid.enabled = True
viewer.reset_view()
print("\nProceed to Napari window to view your selected image.")

-----

## **EXTRACTION**

## **`STEP 1` - Segment nuclei**
### **`A. Select channel`**

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify which channel includes your nuclei label:
- `NUC_CH`: the index of the channel containing your nuclei label. Image indexing begins with 0, not 1. Reference the channel numbers indicated in the Napari window for easy reference.

In [ ]:
#### USER INPUT REQUIRED ###
NUC_CH = 2

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block extracts the nuclei channel from your multi-channel image. It will be the only part of the image used in the rest of this section of the workflow. The single nuclei channel is added to the Napari viewer.

In [ ]:
# select channel
raw_nuclei = select_channel_from_raw(img_data, NUC_CH)

# clear napari and add single channel as a new layer
viewer.layers.clear()
viewer.grid.enabled = False
viewer.add_image(raw_nuclei, scale=scale, name="1-Nuc: Extract Channel")

### **`B. Rescale and smooth image`**

&#x1F453; **FYI:** This code block rescales the image so that the pixel/voxel with the highest intensity is set to 1 and the one with the lowest intensity is set to 0. The image is then *optionally* smoothed using a Gaussian and/or median filter. 

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the amount of filter to use for each method. Higher values indicate more smoothing:
- `med_filter_size`: the size of the median filter to apply; if 0 is used, no filter will be applied
- `gaussian_smoothing_sigma`: the sigma to apply in the Gaussian filtering step; if 0 is used, no filter will be applied

In [ ]:
#### USER INPUT REQUIRED ###
nuc_med_filter_size = 3
nuc_gaussian_smoothing_sigma = 3

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block rescales the image and applies the specified median and Gaussian filters. The image is then added to Napari as a new layer for visual comparison to the input image. 

Use the Napari viewer to iteratively adjust the smoothing settings selected above.

In [ ]:
# rescaling and smoothing input image
nuclei =  scale_and_smooth(raw_nuclei,
                           median_size = nuc_med_filter_size, 
                           gauss_sigma = nuc_gaussian_smoothing_sigma)

# adding image to Napari as a new layer
viewer.add_image(nuclei, scale=scale, name=f"1-Nuc: Smooth - med {nuc_med_filter_size}, gauss {nuc_gaussian_smoothing_sigma}")

### **`C. Apply log transform and threshold`**

&#x1F453; **FYI:** This code block applies a log transform and creates semantic segmentation using [Li's Minimum Cross Entropy](https://scikit-image.org/docs/stable/api/skimage.filters.html#skimage.filters.threshold_li) method. `Semantic segmentation` is the process of deciding whether a pixel/voxel should be included in an object (labeled with a value of 1) or should be considered as part of the background (labeled with a value of 0). A semantic segmentation does not discern individual objects from one another.


#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following information:
- `threshold_factor`: adjustment to make to the local threshold; larger values make the segmentation more stringent (less area included)
- `thresh_min`: minimum bound of the threshold
- `thresh_max`: maximum bound of the threshold

In [ ]:
#### USER INPUT REQUIRED ###
nuc_threshold_factor = 0.9
nuc_thresh_min = 0.09
nuc_thresh_max = 1.0

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block executes the log transform and threshold using the settings above.

Use the Napari viewer to iteratively adjust the filter settings as needed.

In [ ]:
# log transform the image, calculate the threshold value using Li minimum cross entropy method, inverse log transform the value
li_thresholded = apply_log_li_threshold(nuclei, 
                                        thresh_factor=nuc_threshold_factor, 
                                        thresh_min=nuc_thresh_min, 
                                        thresh_max=nuc_thresh_max)

# adding image to Napari as a new layer
viewer.add_image(li_thresholded, scale=scale, name=f"1-Nuc: Threshold - factor {nuc_threshold_factor}, min {nuc_thresh_min}, max {nuc_thresh_max}", opacity=0.3, colormap="cyan", blending='additive')

### **`D. Remove small holes and objects`**

&#x1F453; **FYI:** This code block cleans up the semantic segmentation by filling small holes and/or removing small objects that can be considered errors in the initial segmentation. 

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following values:
- `hole_min_width`: the width of the smallest hole to be filled
- `hole_max_width`: the width of the largest hole to be filled
- `small_object_width`: the width of the largest object to be removed; any object smaller than this size will be removed
- `fill_filter_method`: "3D" processes the image taking into account segmentation values in three dimensions (XYZ); "slice_by_slice" processes each Z-slice in the image separately, not considering the segmentation results in higher or lower Z planes.

In [ ]:
#### USER INPUT REQUIRED ###
nuc_hole_min_width = 0
nuc_hole_max_width = 5

nuc_small_object_width = 25

nuc_fill_filter_method = "slice_by_slice" # "slice_by_slice" or "3D"

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block executes the dot filter using the settings above.

Use the Napari viewer to iteratively adjust the filter settings as needed. 

*Hint: white pixels/voxels are the ones remaining after this step*

In [ ]:
# combine the above functions into one for downstream use in plugin
cleaned_img = fill_and_filter_linear_size(li_thresholded, 
                                           hole_min=nuc_hole_min_width, 
                                           hole_max=nuc_hole_max_width, 
                                           min_size=nuc_small_object_width,
                                           method=nuc_fill_filter_method)

# adding image to Napari as a new layer
viewer.add_image(cleaned_img, scale=scale, name=f"1-Nuc: Fill holes {nuc_hole_min_width}-{nuc_hole_max_width} and remove small objects {nuc_small_object_width}", colormap="magenta", blending="additive")

### **`E. - Label objects`**

#### &#x1F3C3; **Run code; no user input required**
&#x1F453; **FYI:** This code block takes the semantic segmentation and creates an `instance segmentation`. In this output, each individual object in the image is given a unique ID number. The background pixels/voxels are still labeled as 0, but now each pixle/voxel within an object is labeled as a positive integer. 

In this workflow objects are first separated based on connectivity: if a pixel/voxel is touching another pixel/voxel in any direction, they are considered the same object and each pixel/voxel within that object is labeled as the same unique ID number. 

*In the Napari viewer, the image is added as a "labels" layer where each object appears as a different color.*

In [ ]:
# create instance segmentation based on connectivity
nuclei_labels = label_uint16(cleaned_img)

# adding image to Napari as a new layer
viewer.add_labels(nuclei_labels, scale=scale, name="1-Nuc: Instance segmentation")

#### &#x1F6D1; &#x270D; **User Input Required:**

&#x1F453; **FYI:** This code block is used to further separate the nuclei objects using a `watershed` algorithm.

Specify the inputs below to indicate if your nuclei segmentation was sufficient to separate individual and how further declupming should be done if needed:
- `declumping`: False indicates that the nuclei objects have been properly separated from one another (no or almost no clumps of multiple nuclei should remain) and declumping will not be preformed; True indicates that further nuclei separation ("declumping") will be carried out with the specified settings.
- `footprint_sz`: the size of the isotropic footprint used to erode the nuclei segmentation when creating "seeds" for the watershed; the goal is to produce one seed per nuclei through erosion. Large values will erode objects more, possibly resulting in more objects being separated during watershed. However, if this value is too large, you may begin to lose objects
- `seed_small_object_size`: after the nuclei are eroded, small objects are removed to safe guard against multiple seeds being produced per nuclei object. Increase this value is multiple seeds are being generated per nuclei or the nuclei are being over separated.

In [ ]:
#### USER INPUT REQUIRED ###
declumping = True
footprint_sz = 80 # used to define the area that local maxima are found within
seed_small_object_size = 2

#### &#x1F3C3; **Run code; no user input required**
&#x1F453; **FYI:** This code block durther refines the labeled segementation created above with the goal of separating individual nuclei.

In [ ]:
# apply a watershed separate nuclei objects further; eroded nuclei labels are used as markers
if declumping==False:
    nuclei_labels_dc = nuclei_labels
    print("Nuclei are already separated; no declumping applied.")
elif declumping==True:
    # declumping with watershed using eroded nuclei labels as markers
    inverse = np.max(nuclei) - nuclei
    
    # binary erosion approach to ID seeds (fooitprint = 15 is good ish)
    seeds = binary_erosion(nuclei_labels, footprint=[(np.ones((footprint_sz, 1, 1)), 1), (np.ones((1, footprint_sz, 1)), 1), (np.ones((1, 1, footprint_sz)), 1)])
    seeds_refined = label(fill_and_filter_linear_size(seeds, hole_min=0, hole_max=0, min_size=seed_small_object_size, method="3D"))
    nuclei_labels_dc = watershed(inverse,
                                 markers=seeds_refined,
                                 connectivity=2,
                                 mask=nuclei_labels)
    # adding image to Napari as a new layer
    viewer.add_labels(seeds_refined, scale=scale, name=f"1-Nuc: Seeds for watershed declumping - small object size for seeds {seed_small_object_size}")


    # distance transform approach to ID seeds (footprint = 80 is good)
    # distance = ndi.distance_transform_edt(nuclei_labels, sampling=scale)
    # # new_dist = (distance*1000).astype(int)  # convert to int after multiplying by 1000 to preserve decimal information; needs to be int for peak_local_max function
    # multiply = (distance * nuclei)/1000
    # # hard footprint based on ZYX scale format
    # coords = peak_local_max(multiply, footprint=np.ones((round((footprint_sz*scale[0])/scale[1]), footprint_sz, footprint_sz)), labels=nuclei_labels)
    # mask = np.zeros(distance.shape, dtype=bool)
    # mask[tuple(coords.T)] = True
    # markers, _ = ndi.label(mask)

    # nuclei_labels_dc = watershed(inverse, markers, mask=nuclei_labels)

    # viewer.add_image(multiply, scale=scale, name="1-Nuc: Distance transform + nuclei intensity", colormap="magma", blending="additive")
    # viewer.add_labels(markers, scale=scale, name=f"1-Nuc: Seeds for watershed declumping - small object size for seeds {seed_small_object_size}")

        
    print("Declumping applied. View output in Napari viewer.")
else:
    raise ValueError("nuclei_separated must be either True or False")

viewer.add_labels(nuclei_labels_dc, scale=scale, name=f"1-Nuc: Declumped (declumping={declumping}, footprint size={footprint_sz}, seed small object size={seed_small_object_size})")

### **`F. - Filter objects`**

#### &#x1F6D1; &#x270D; **User Input Required**
&#x1F453; **FYI:** This step will filter out nuclei that are brighter than the average nuclei in the image. These nuclei are abnormally rounded, possibly due to division or cell death.

Please specify the following:
- `nuc_intensity_threshold`: the average intensity value to use as a treshold cut off when removing nuclei objects; any objects with average intensities above this value will be excluded from the final segmentation image.

In [ ]:
### USER INPUT REQUIRED ###
nuc_intensity_threshold = 6000

#### &#x1F3C3; **Run code; no user input required**
&#x1F453; **FYI:** This code block filters the nuclei objects based on their intensities - any nuclei that have an average intensity above the threshold indicated will be removed from the instance segmentation image.

In [ ]:
nuclei_labels_filtered = filter_nuc_by_intensity(nuclei_labels_dc, raw_nuclei, intensity_threshold=nuc_intensity_threshold)

print("Before filtering, there were: ", len(np.unique(nuclei_labels_dc))-1, "nuclei.")
print("After filtering, there are: ", len(np.unique(nuclei_labels_filtered))-1, "nuclei.")

viewer.add_labels(nuclei_labels_filtered, scale=scale, name=f"9-Nuc: removed nuclei with mean intensity above {nuc_intensity_threshold}", opacity=1)

## **`STEP 2` - Create composite image**

&#x1F453; **FYI:** This code block creates a composite image of the organelle channels. The intensity of each channel is combined  with specified weights. The resulting composite will be used to segment the cell area. 

*Hint: aim to combine channels in a way that produces a relatively even intensity throughout the cell.* 

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following values:
- `w0`: the weight of channel 0 (the first channel); a value of 0 will exclude this channel from the compostite; large values will cause this channel to be more prominent in the final composite image
- `w1`: the weight of channel 1
- `w2`: the weight of channel 2
- `w3`: the weight of channel 3
- `w4`: the weight of channel 4
- `w5`: the weight of channel 5
- `w6`: the weight of channel 6
- `w7`: the weight of channel 7
- `w8`: the weight of channel 8
- `w9`: the weight of channel 9
- `rescale`: "True" rescales the image so that the pixel/voxel with the highest intensity is set to 1 and the one with the lowest intensity is set to 0; "False" leaves the combined intensity values as is after the composite is made

In [ ]:
#### USER INPUT REQUIRED ###
w0 = 3
w1 = 2
w2 = 1
w3 = 0
w4 = 0
w5 = 0
w6 = 0
w7 = 0
w8 = 0
w9 = 0
rescale = True

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block creates the composite and optionally rescales the image based on the settings above. The image is then added to Napari as a new layer for visual comparison to the input image. 

Use the Napari viewer to iteratively adjust the smoothing settings selected above.

In [ ]:
# make aggregate and optionally rescale
struct_img_raw = make_aggregate(img_data,
               weight_ch0= w0,
               weight_ch1= w1,
               weight_ch2= w2,
               weight_ch3= w3,
               weight_ch4= w4,
               weight_ch5= w5,
               weight_ch6= w6,
               weight_ch7= w7,
               weight_ch8= w8,
               weight_ch9= w9,
               rescale = rescale)

# adding image to Napari as a new layer
viewer.layers.clear()
viewer.add_labels(nuclei_labels_dc, scale=scale, name=f"1-Nuc: FINAL OUTPUT")
viewer.add_image(img_data, scale=scale, name="Raw image")
viewer.add_image(struct_img_raw, scale=scale, name="2-Cell: Create composite")

-----
## **PRE-PROCESSING**

## **`STEP 3` - Rescale and smooth image**

&#x1F453; **FYI:** This code block rescales the image so that the pixel/voxel with the highest intensity is set to 1 and the one with the lowest intensity is set to 0. The image is then *optionally* smoothed using a Gaussian and/or median filter. 

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the amount of filter to use for each method. Higher values indicate more smoothing:
- `med_filter_size`: the size of the median filter to apply; if 0 is used, no filter will be applied
- `gaussian_smoothing_sigma`: the sigma to apply in the Gaussian filtering step; if 0 is used, no filter will be applied

In [ ]:
#### USER INPUT REQUIRED ###
med_filter_size = 3
gaussian_smoothing_sigma = 3

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block rescales the image and applies the specified median and Gaussian filters. The image is then added to Napari as a new layer for visual comparison to the input image. 

Use the Napari viewer to iteratively adjust the smoothing settings selected above.

In [ ]:
# rescaling and smoothing input image
structure_img_smooth = scale_and_smooth(struct_img_raw,
                                        median_size = med_filter_size, 
                                        gauss_sigma = gaussian_smoothing_sigma)

# adding image to Napari as a new layer
viewer.add_image(structure_img_smooth, scale=scale, name=f"3-Cell: Rescale and Smooth; Gaus={gaussian_smoothing_sigma}, med={med_filter_size}")

## **`STEP 4` - Log transform + Scharr edge detection**

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block applies a log transform and the [Scharr](https://scikit-image.org/docs/stable/api/skimage.filters.html#skimage.filters.scharr) edge detection filter.

Use the Napari viewer to iteratively adjust the filter settings as needed.

*Hint: it may be helpful to readjust the composite and smoothing settings to improve the outcome of this step.*

In [ ]:
# log scale the image, apply the scharr edge detection filter to logged image, add the two images together
composite_cellmask = non_linear_cellmask_transform(structure_img_smooth)

# adding image to Napari as a new layer
viewer.add_image(composite_cellmask, scale=scale, name="4-Cell: Log Scharr edge detectiong")

-----
## **CORE-PROCESSING**

## **`STEP 5` - Global + local thresholding (AICSSeg – MO)**

&#x1F453; **FYI:** This code block creates a semantic segmentation of the cell area. `Semantic segmentation` is the process of deciding whether a pixel/voxel should be included in an object (labeled with a value of 1) or should be considered as part of the background (labeled with a value of 0). A semantic segmentation does not discern individual objects from one another.

The masked_object_filter utilizes the 'MO' filter from the [`aics-segmentation`](https://github.com/AllenCell/aics-segmentation) package. AICS documentation states: "The algorithm is a hybrid thresholding method combining two levels of thresholds. The steps are: [1] a global threshold is calculated, [2] extract each individual connected componet after applying the global threshold, [3] remove small objects, [4] within each remaining object, a local Otsu threshold is calculated and applied with an optional local threshold adjustment ratio (to make the segmentation more and less conservative). An extra check can be used in step [4], which requires the local Otsu threshold larger than 1/3 of global Otsu threhsold and otherwise this connected component is discarded."

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following values:
- `thresh_method`: the global thresholding method; options include 'tri'/'triangle, 'med'/'median' or 'ave'/'ave_tri_med'. Triangle implements the [skimage.filters.threshold_triangle](https://scikit-image.org/docs/stable/api/skimage.filters.html#skimage.filters.threshold_triangle) method, median utilizes the 50th percentile of the intensity values in the image, and ave uses the average of the two methods to calculate the lower bound of the global threshold.
- `cell_wise_min_area`: the minimum expected size of your object; smaller objects will be removed prior to local thresholding
- `thresh_adj`: adjustment to make to the local threshold; larger values make the segmentation more stringent (less area included)

In [ ]:
#### USER INPUT REQUIRED ###
thresh_method = 'med'
cutoff_size =  500
thresh_adj = 0.5

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block executes the MO filter using the settings above. *Note: this step can take a long time with lower cutoff values.*

Use the Napari viewer to iteratively adjust the filter settings as needed.

In [ ]:
# threshold the composite image after log/edge detection using the MO filter function from aicssegmentation - this applies a global threshold, then a local threshold to produce a semantic segmentation
bw = masked_object_thresh(composite_cellmask, 
                          global_method=thresh_method, 
                          cutoff_size=cutoff_size, 
                          local_adjust=thresh_adj)

# adding image to Napari as a new layer
viewer.add_image(bw, scale=scale, name=f"5-Cell: MO filter; method={thresh_method}, cutoff={cutoff_size}, adjust={thresh_adj}", opacity=0.3, colormap="cyan", blending='additive')

-----
## **POST-PROCESSING**

## **`STEP 6` - Remove small holes and objects**

&#x1F453; **FYI:** This code block cleans up the semantic segmentation by filling small holes and/or removing small objects that can be considered errors in the initial segmentation. 

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following values:
- `hole_min_width`: the width of the smallest hole to be filled
- `hole_max_width`: the width of the largest hole to be filled
- `small_object_width`: the width of the largest object to be removed; any object smaller than this size will be removed
- `fill_filter_method`: "3D" processes the image taking into account segmentation values in three dimensions (XYZ); "slice-by-slice" processes each Z-slice in the image separately, not considering the segmentation results in higher or lower Z planes.

In [ ]:
#### USER INPUT REQUIRED ###
hole_min_width = 0
hole_max_width = 60

small_object_width = 45

method = 'slice_by_slice'

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block fills small holes and removes small objects using the settings above.

Use the Napari viewer to iteratively adjust the filter settings as needed. 

*Hint: white pixels/voxels are the ones remaining after this step*

In [ ]:
# fill small holes and remove small objects
cleaned_img2 = fill_and_filter_linear_size(bw, 
                                           hole_min=hole_min_width, 
                                           hole_max=hole_max_width, 
                                           min_size= small_object_width,
                                           method = method)

# adding image to Napari as a new layer
viewer.add_image(cleaned_img2, scale=scale, name=f"6-Cell: Fill holes ({hole_min_width}-{hole_max_width}), Remove small objects (<{small_object_width})", colormap="magenta", blending="additive")

-----
## **POST-POST-PROCESSING**

## **`STEP 7` - Separate individual cells based on nuclei segmentation**

&#x1F453; **FYI:** This code block takes the semantic segmentation of the cell and creates an `instance segmentation`. The process is accomplished with the [watershed](https://scikit-image.org/docs/stable/api/skimage.segmentation.html#skimage.segmentation.watershed) function where the nuclei are used as seeds to determine the approximate center of each cell. 

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following values:
- `watershed_method`: "3D" processes the image taking into account segmentation values in three dimensions (XYZ); "slice-by-slice" processes each Z-slice in the image separately, not considering the segmentation results in higher or lower Z planes.

In [ ]:
#### USER INPUT REQUIRED ###
watershed_method = '3D'

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block executes the watershed and cell selection process using the settings above.

Use the Napari viewer to iteratively adjust the filter settings as needed. 

*In the Napari viewer, the image is added as a "labels" layer where each object appears as a different color. There should only be a single cell remaining at this stage.*

In [ ]:
# apply a watershed to the inverted image using the nuclei as a seed for each cell
cellmask_labels = masked_inverted_watershed(structure_img_smooth, 
                                            nuclei_labels, 
                                            cleaned_img2,
                                            method=watershed_method)

# adding image to Napari as a new layer
viewer.add_image(struct_img_raw, scale=scale)
viewer.add_labels(nuclei_labels_dc, scale=scale, name=f"1-Nuc: FINAL OUTPUT")
viewer.add_labels(cellmask_labels, scale=scale, name="7-Cell: FINAL OUTPUT")

## **`STEP 8` - Stack masks**

#### &#x1F3C3; **Run code; no user input required**
&#x1F453; **FYI:** This code block stacks the single nucleus and cell masks into one multichannel image.

In [ ]:
stack = np.stack([cellmask_labels, nuclei_labels_dc], axis=0).astype(np.uint8)
print(f"Stack mask file structure: {np.shape(stack)}")
print(f"The first dimension (value '2') represents the 'cell' and 'nucleus' channels, in that order.")

-----
## **SAVING**

## **`Saving` - Save the segmentation output**

#### &#x1F3C3; **Run code; no user input required**
&#x1F453; **FYI:** This code block saves the instance segmentation output to the `out_data_path` specified earlier. You don't need to do this for each individual image unless you want a record of what the segmentation looks like at this point. As long as the image is included in one of your batch processing runs, it will be processed and saved then.

In [ ]:
# Saving file
tifffile.imwrite(f"{str(seg_data_path / Path(file_path).stem)}-masks.tiff", stack.astype(np.uint16), metadata=meta_dict)
print(f"saved to: {seg_data_path}")

### **`Exporting settings` - Print out of the settings selected above for batch processing**

#### &#x1F3C3; **Run code; no user input required**
&#x1F453; **FYI:** This code block reformats and prints your selected settings from above. The resulting output (found below the cell) can be copied and pasted into the `batch_process_segmentation` function.

In [ ]:
[NUC_CH,
nuc_med_filter_size,
nuc_gaussian_smoothing_sigma,
nuc_threshold_factor,
nuc_thresh_min,
nuc_thresh_max,
nuc_hole_min_width,
nuc_hole_max_width,
nuc_small_object_width,
nuc_fill_filter_method,
declumping,
footprint_sz,
seed_small_object_size,
nuc_intensity_threshold,
[w0, w1, w2, w3, w4, w5, w6, w7, w8, w9],
rescale,
med_filter_size,
gaussian_smoothing_sigma,
thresh_method,
thresh_adj,
cutoff_size,
hole_min_width,
hole_max_width,
small_object_width,
method,
watershed_method]

-----
-----
## **Utilizing the `infer_masks` functions**

If you would like to process all of the steps listed above on your selected image, you can use this function. This will be the same function applied to each image during batch processing.

#### &#x1F3C3; **Run code; no user input required**
&#x1F453; **FYI:** This code block defines the `infer_masks()`, function, the complete function that combines the three above and that will be used for batch processing. It is applied below.

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block applies the function above to your test image. The settings specified above are applied here.

In [ ]:
# run nuclei segmentation from label
stacked_masks = infer_masks(img_data, 
                              NUC_CH,
                              nuc_med_filter_size,
                              nuc_gaussian_smoothing_sigma,
                              nuc_threshold_factor,
                              nuc_thresh_min,
                              nuc_thresh_max,
                              nuc_hole_min_width,
                              nuc_hole_max_width,
                              nuc_small_object_width,
                              nuc_fill_filter_method,
                              declumping,
                              footprint_sz,
                              seed_small_object_size,
                              nuc_intensity_threshold,
                              [w0, w1, w2, w3, w4, w5, w6, w7, w8, w9],
                              rescale,
                              med_filter_size,
                              gaussian_smoothing_sigma,
                              thresh_method,
                              thresh_adj,
                              cutoff_size,
                              hole_min_width,
                              hole_max_width,
                              small_object_width,
                              method,
                              watershed_method)

#confirm this output matches the output saved above
print(f"The segmentation output here matches the output created above: {np.all(stack == stacked_masks)}")

-------------
### ✅ **INFER CELL AND NUCLEUS MASKS COMPLETE!**

Continue to the [`batch_process_segmentation`](batch_process_segmentations.ipynb) notebook where you can apply the above settings to many images.